In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)

In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DATA_PATH = PROJECT_ROOT / "01_data" / "processed" / "daily_sku_sales_processed.csv"
FEATURE_DATA_PATH = PROJECT_ROOT / "01_data" / "features" / "sales_features.csv"

FEATURE_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

print(PROCESSED_DATA_PATH)
print(FEATURE_DATA_PATH)

..\01_data\processed\daily_sku_sales_processed.csv
..\01_data\features\sales_features.csv


In [3]:
df = pd.read_csv(PROCESSED_DATA_PATH)

df.head()

,shipped_date,sku,qty,revenue,cost_of_good_sold,moq_order,channel_count
0,2021-01-01,089A0E,220,5821.2,2926.0,56460,2
1,2021-01-03,089A0E,140,3704.4,2156.0,56460,1
2,2021-01-05,089A0E,150,3969.0,2310.0,56460,1
3,2021-01-09,089A0E,190,5027.4,2926.0,56460,1
4,2021-01-11,089A0E,60,1587.6,924.0,56460,1


In [4]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum())

print("\nData types:")
display(df.dtypes)

Shape: (29004, 7)

Columns:
['shipped_date', 'sku', 'qty', 'revenue', 'cost_of_good_sold', 'moq_order', 'channel_count']

Missing values:


shipped_date         0
sku                  0
qty                  0
revenue              0
cost_of_good_sold    0
moq_order            0
channel_count        0
dtype: int64


Data types:


shipped_date             str
sku                      str
qty                    int64
revenue              float64
cost_of_good_sold    float64
moq_order              int64
channel_count          int64
dtype: object

In [5]:
df["shipped_date"] = pd.to_datetime(df["shipped_date"])

df = df.sort_values(["sku", "shipped_date"]).reset_index(drop=True)

df.head()

,shipped_date,sku,qty,revenue,cost_of_good_sold,moq_order,channel_count
0,2021-01-01,089A0E,220,5821.2,2926.0,56460,2
1,2021-01-03,089A0E,140,3704.4,2156.0,56460,1
2,2021-01-05,089A0E,150,3969.0,2310.0,56460,1
3,2021-01-09,089A0E,190,5027.4,2926.0,56460,1
4,2021-01-11,089A0E,60,1587.6,924.0,56460,1


In [6]:
df["year"] = df["shipped_date"].dt.year
df["month"] = df["shipped_date"].dt.month
df["day"] = df["shipped_date"].dt.day
df["day_of_week"] = df["shipped_date"].dt.dayofweek
df["week_of_year"] = df["shipped_date"].dt.isocalendar().week.astype(int)
df["quarter"] = df["shipped_date"].dt.quarter
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

df[[
    "shipped_date", 
    "year", 
    "month", 
    "day", 
    "day_of_week", 
    "week_of_year", 
    "quarter", 
    "is_weekend"
]].head()

,shipped_date,year,month,day,day_of_week,week_of_year,quarter,is_weekend
0,2021-01-01,2021,1,1,4,53,1,0
1,2021-01-03,2021,1,3,6,53,1,1
2,2021-01-05,2021,1,5,1,1,1,0
3,2021-01-09,2021,1,9,5,1,1,1
4,2021-01-11,2021,1,11,0,2,1,0


In [7]:
lag_periods = [1, 2, 3, 7, 14]

for lag in lag_periods:
    df[f"qty_lag_{lag}"] = df.groupby("sku")["qty"].shift(lag)

df[[
    "shipped_date", 
    "sku", 
    "qty", 
    "qty_lag_1", 
    "qty_lag_2", 
    "qty_lag_3", 
    "qty_lag_7", 
    "qty_lag_14"
]].head(20)

,shipped_date,sku,qty,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_7,qty_lag_14
0,2021-01-01,089A0E,220,NaN,NaN,NaN,NaN,NaN
1,2021-01-03,089A0E,140,220.0,NaN,NaN,NaN,NaN
2,2021-01-05,089A0E,150,140.0,220.0,NaN,NaN,NaN
3,2021-01-09,089A0E,190,150.0,140.0,220.0,NaN,NaN
4,2021-01-11,089A0E,60,190.0,150.0,140.0,NaN,NaN
5,2021-01-15,089A0E,140,60.0,190.0,150.0,NaN,NaN
6,2021-01-17,089A0E,50,140.0,60.0,190.0,NaN,NaN
7,2021-01-19,089A0E,60,50.0,140.0,60.0,220.0,NaN
8,2021-01-21,089A0E,190,60.0,50.0,140.0,140.0,NaN
9,2021-01-23,089A0E,60,190.0,60.0,50.0,150.0,NaN


In [8]:
rolling_windows = [3, 7, 14]

for window in rolling_windows:
    df[f"qty_rolling_mean_{window}"] = (
        df.groupby("sku")["qty"]
        .transform(lambda x: x.shift(1).rolling(window=window).mean())
    )
    
    df[f"qty_rolling_std_{window}"] = (
        df.groupby("sku")["qty"]
        .transform(lambda x: x.shift(1).rolling(window=window).std())
    )
    
    df[f"qty_rolling_min_{window}"] = (
        df.groupby("sku")["qty"]
        .transform(lambda x: x.shift(1).rolling(window=window).min())
    )
    
    df[f"qty_rolling_max_{window}"] = (
        df.groupby("sku")["qty"]
        .transform(lambda x: x.shift(1).rolling(window=window).max())
    )

df.head()

,shipped_date,sku,qty,revenue,cost_of_good_sold,moq_order,channel_count,year,month,day,day_of_week,week_of_year,quarter,is_weekend,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_7,qty_lag_14,qty_rolling_mean_3,qty_rolling_std_3,qty_rolling_min_3,qty_rolling_max_3,qty_rolling_mean_7,qty_rolling_std_7,qty_rolling_min_7,qty_rolling_max_7,qty_rolling_mean_14,qty_rolling_std_14,qty_rolling_min_14,qty_rolling_max_14
0,2021-01-01,089A0E,220,5821.2,2926.0,56460,2,2021,1,1,4,53,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-03,089A0E,140,3704.4,2156.0,56460,1,2021,1,3,6,53,1,1,220.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-05,089A0E,150,3969.0,2310.0,56460,1,2021,1,5,1,1,1,0,140.0,220.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-01-09,089A0E,190,5027.4,2926.0,56460,1,2021,1,9,5,1,1,1,150.0,140.0,220.0,NaN,NaN,170.0,43.588989,140.0,220.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-01-11,089A0E,60,1587.6,924.0,56460,1,2021,1,11,0,2,1,0,190.0,150.0,140.0,NaN,NaN,160.0,26.457513,140.0,190.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df["sku_expanding_mean_qty"] = (
    df.groupby("sku")["qty"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

df["sku_expanding_median_qty"] = (
    df.groupby("sku")["qty"]
    .transform(lambda x: x.shift(1).expanding().median())
)

df["sku_expanding_std_qty"] = (
    df.groupby("sku")["qty"]
    .transform(lambda x: x.shift(1).expanding().std())
)

df[[
    "shipped_date",
    "sku",
    "qty",
    "sku_expanding_mean_qty",
    "sku_expanding_median_qty",
    "sku_expanding_std_qty"
]].head(20)

,shipped_date,sku,qty,sku_expanding_mean_qty,sku_expanding_median_qty,sku_expanding_std_qty
0,2021-01-01,089A0E,220,NaN,NaN,NaN
1,2021-01-03,089A0E,140,220.000000,220.0,NaN
2,2021-01-05,089A0E,150,180.000000,180.0,56.568542
3,2021-01-09,089A0E,190,170.000000,150.0,43.588989
4,2021-01-11,089A0E,60,175.000000,170.0,36.968455
5,2021-01-15,089A0E,140,152.000000,150.0,60.580525
6,2021-01-17,089A0E,50,150.000000,145.0,54.405882
7,2021-01-19,089A0E,60,135.714286,140.0,62.411843
8,2021-01-21,089A0E,190,126.250000,140.0,63.681686
9,2021-01-23,089A0E,60,133.333333,140.0,63.245553


In [10]:
df["sku_encoded"] = df["sku"].astype("category").cat.codes

sku_mapping = (
    df[["sku", "sku_encoded"]]
    .drop_duplicates()
    .sort_values("sku_encoded")
    .reset_index(drop=True)
)

sku_mapping.head()

,sku,sku_encoded
0,089A0E,0
1,0FKFLA,1
2,0NFJ14,2
3,13KMDL,3
4,17KYMH,4


In [11]:
df["qty_log"] = np.log1p(df["qty"])

df[["qty", "qty_log"]].head()

,qty,qty_log
0,220,5.398163
1,140,4.948760
2,150,5.017280
3,190,5.252273
4,60,4.110874


In [12]:
missing_summary = df.isna().sum().sort_values(ascending=False)

missing_summary[missing_summary > 0]

qty_rolling_mean_14         2520
qty_rolling_max_14          2520
qty_rolling_min_14          2520
qty_lag_14                  2520
qty_rolling_std_14          2520
qty_rolling_std_7           1260
qty_rolling_mean_7          1260
qty_rolling_max_7           1260
qty_rolling_min_7           1260
qty_lag_7                   1260
qty_lag_3                    540
qty_rolling_mean_3           540
qty_rolling_std_3            540
qty_rolling_max_3            540
qty_rolling_min_3            540
qty_lag_2                    360
sku_expanding_std_qty        360
sku_expanding_mean_qty       180
qty_lag_1                    180
sku_expanding_median_qty     180
dtype: int64

In [13]:
df_features = df.dropna().reset_index(drop=True)

print("Before dropping missing:", df.shape)
print("After dropping missing:", df_features.shape)

df_features.head()

Before dropping missing: (29004, 36)
After dropping missing: (26484, 36)


,shipped_date,sku,qty,revenue,cost_of_good_sold,moq_order,channel_count,year,month,day,day_of_week,week_of_year,quarter,is_weekend,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_7,qty_lag_14,qty_rolling_mean_3,qty_rolling_std_3,qty_rolling_min_3,qty_rolling_max_3,qty_rolling_mean_7,qty_rolling_std_7,qty_rolling_min_7,qty_rolling_max_7,qty_rolling_mean_14,qty_rolling_std_14,qty_rolling_min_14,qty_rolling_max_14,sku_expanding_mean_qty,sku_expanding_median_qty,sku_expanding_std_qty,sku_encoded,qty_log
0,2021-02-04,089A0E,20,529.2,428.65,56460,1,2021,2,4,3,5,1,0,300.0,110.0,90.0,60.0,220.0,166.666667,115.902258,90.0,300.0,121.428571,92.992575,40.0,300.0,128.571429,76.445772,40.0,300.0,128.571429,125.0,76.445772,0,3.044522
1,2021-02-06,089A0E,60,1587.6,1285.96,56460,1,2021,2,6,5,5,1,1,20.0,300.0,110.0,190.0,140.0,143.333333,142.945211,20.0,300.0,115.714286,98.464400,20.0,300.0,114.285714,76.732732,20.0,300.0,121.333333,110.0,78.818659,0,4.110874
2,2021-02-10,089A0E,40,1058.4,857.30,56460,1,2021,2,10,2,6,1,0,60.0,20.0,300.0,60.0,150.0,126.666667,151.437556,20.0,300.0,97.142857,94.289322,20.0,300.0,108.571429,77.643876,20.0,300.0,117.500000,100.0,77.674535,0,3.713572
3,2021-02-12,089A0E,100,2646.0,2143.26,56460,1,2021,2,12,4,6,1,0,40.0,60.0,20.0,40.0,190.0,40.000000,20.000000,20.0,60.0,94.285714,95.891804,20.0,300.0,100.714286,78.687726,20.0,300.0,112.941176,90.0,77.521344,0,4.615121
4,2021-02-14,089A0E,50,1323.0,1071.63,56460,1,2021,2,14,6,6,1,1,100.0,40.0,60.0,90.0,60.0,66.666667,30.550505,40.0,100.0,102.857143,92.864469,20.0,300.0,94.285714,74.391303,20.0,300.0,112.222222,95.0,75.268582,0,3.931826


In [14]:
print("Shape:", df_features.shape)
print("Unique SKUs:", df_features["sku"].nunique())
print("Date range:", df_features["shipped_date"].min(), "to", df_features["shipped_date"].max())

print("\nMissing values after cleaning:")
display(df_features.isna().sum().sum())

print("\nColumns:")
display(df_features.columns.tolist())

Shape: (26484, 36)
Unique SKUs: 180
Date range: 2021-01-29 00:00:00 to 2022-01-02 00:00:00

Missing values after cleaning:


np.int64(0)


Columns:


['shipped_date',
 'sku',
 'qty',
 'revenue',
 'cost_of_good_sold',
 'moq_order',
 'channel_count',
 'year',
 'month',
 'day',
 'day_of_week',
 'week_of_year',
 'quarter',
 'is_weekend',
 'qty_lag_1',
 'qty_lag_2',
 'qty_lag_3',
 'qty_lag_7',
 'qty_lag_14',
 'qty_rolling_mean_3',
 'qty_rolling_std_3',
 'qty_rolling_min_3',
 'qty_rolling_max_3',
 'qty_rolling_mean_7',
 'qty_rolling_std_7',
 'qty_rolling_min_7',
 'qty_rolling_max_7',
 'qty_rolling_mean_14',
 'qty_rolling_std_14',
 'qty_rolling_min_14',
 'qty_rolling_max_14',
 'sku_expanding_mean_qty',
 'sku_expanding_median_qty',
 'sku_expanding_std_qty',
 'sku_encoded',
 'qty_log']

In [15]:
base_cols = [
    "shipped_date",
    "sku",
    "sku_encoded",
    "qty",
    "qty_log"
]

date_cols = [
    "year",
    "month",
    "day",
    "day_of_week",
    "week_of_year",
    "quarter",
    "is_weekend"
]

business_cols = [
    "revenue",
    "cost_of_good_sold",
    "moq_order",
    "channel_count"
]

lag_cols = [col for col in df_features.columns if col.startswith("qty_lag_")]
rolling_cols = [col for col in df_features.columns if col.startswith("qty_rolling_")]
sku_history_cols = [col for col in df_features.columns if col.startswith("sku_expanding_")]

final_cols = (
    base_cols
    + date_cols
    + business_cols
    + lag_cols
    + rolling_cols
    + sku_history_cols
)

df_features = df_features[final_cols]

df_features.head()

,shipped_date,sku,sku_encoded,qty,qty_log,year,month,day,day_of_week,week_of_year,quarter,is_weekend,revenue,cost_of_good_sold,moq_order,channel_count,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_7,qty_lag_14,qty_rolling_mean_3,qty_rolling_std_3,qty_rolling_min_3,qty_rolling_max_3,qty_rolling_mean_7,qty_rolling_std_7,qty_rolling_min_7,qty_rolling_max_7,qty_rolling_mean_14,qty_rolling_std_14,qty_rolling_min_14,qty_rolling_max_14,sku_expanding_mean_qty,sku_expanding_median_qty,sku_expanding_std_qty
0,2021-02-04,089A0E,0,20,3.044522,2021,2,4,3,5,1,0,529.2,428.65,56460,1,300.0,110.0,90.0,60.0,220.0,166.666667,115.902258,90.0,300.0,121.428571,92.992575,40.0,300.0,128.571429,76.445772,40.0,300.0,128.571429,125.0,76.445772
1,2021-02-06,089A0E,0,60,4.110874,2021,2,6,5,5,1,1,1587.6,1285.96,56460,1,20.0,300.0,110.0,190.0,140.0,143.333333,142.945211,20.0,300.0,115.714286,98.464400,20.0,300.0,114.285714,76.732732,20.0,300.0,121.333333,110.0,78.818659
2,2021-02-10,089A0E,0,40,3.713572,2021,2,10,2,6,1,0,1058.4,857.30,56460,1,60.0,20.0,300.0,60.0,150.0,126.666667,151.437556,20.0,300.0,97.142857,94.289322,20.0,300.0,108.571429,77.643876,20.0,300.0,117.500000,100.0,77.674535
3,2021-02-12,089A0E,0,100,4.615121,2021,2,12,4,6,1,0,2646.0,2143.26,56460,1,40.0,60.0,20.0,40.0,190.0,40.000000,20.000000,20.0,60.0,94.285714,95.891804,20.0,300.0,100.714286,78.687726,20.0,300.0,112.941176,90.0,77.521344
4,2021-02-14,089A0E,0,50,3.931826,2021,2,14,6,6,1,1,1323.0,1071.63,56460,1,100.0,40.0,60.0,90.0,60.0,66.666667,30.550505,40.0,100.0,102.857143,92.864469,20.0,300.0,94.285714,74.391303,20.0,300.0,112.222222,95.0,75.268582


In [16]:
df_features.to_csv(FEATURE_DATA_PATH, index=False)

print(f"Feature dataset saved to: {FEATURE_DATA_PATH}")
print("Final shape:", df_features.shape)

Feature dataset saved to: ..\01_data\features\sales_features.csv
Final shape: (26484, 36)


In [17]:
feature_summary = {
    "feature_rows": df_features.shape[0],
    "feature_columns": df_features.shape[1],
    "unique_skus": df_features["sku"].nunique(),
    "date_min": df_features["shipped_date"].min(),
    "date_max": df_features["shipped_date"].max(),
    "lag_features": len(lag_cols),
    "rolling_features": len(rolling_cols),
    "sku_history_features": len(sku_history_cols)
}

feature_summary_df = pd.DataFrame([feature_summary])
feature_summary_df

,feature_rows,feature_columns,unique_skus,date_min,date_max,lag_features,rolling_features,sku_history_features
0,26484,36,180,2021-01-29,2022-01-02,5,12,3


In [18]:
FEATURE_SUMMARY_PATH = PROJECT_ROOT / "05_outputs" / "model_results" / "feature_summary.csv"

FEATURE_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

feature_summary_df.to_csv(FEATURE_SUMMARY_PATH, index=False)

print(f"Feature summary saved to: {FEATURE_SUMMARY_PATH}")

Feature summary saved to: ..\05_outputs\model_results\feature_summary.csv
